# Métricas e comparação dos modelos YOLOv26

Este notebook cria os gráficos comparativos de YOLOv26 a partir dos artefatos armazenados em `models/`, sem valores de métricas inseridos manualmente.

- **mAP@0.5 global:** melhor valor no `results.csv` de cada variante.
- **mAP@0.5 por classe:** nova validação de cada checkpoint `weights/best.pt` usando o dataset do repositório.

Execute a partir da pasta `notebooks/` ou ajuste `PROJECT_ROOT`.

## 1. Carregar métricas de treinamento registradas

A comparação global lê o melhor mAP@0.5 de validação registrado nos artefatos de cada modelo.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

PROJECT_ROOT = Path("..")
MODELS_DIR = PROJECT_ROOT / "models"
DATA_YAML = PROJECT_ROOT / "dataset" / "data.yaml"
VARIANTS = ("n", "s", "m", "l")
MODEL_LABELS = [f"YOLOv26{variant}" for variant in VARIANTS]
METRIC_COLUMN = "metrics/mAP50(B)"

def artifact_path(variant, filename):
    path = MODELS_DIR / f"yolov26{variant}" / filename
    if not path.exists():
        raise FileNotFoundError(f"Artefato de modelo ausente: {path}")
    return path

def best_map50(variant):
    with artifact_path(variant, "results.csv").open(newline="") as file:
        rows = list(csv.DictReader(file))
    if not rows or METRIC_COLUMN not in rows[0]:
        raise KeyError(f"{METRIC_COLUMN} ausente do results.csv de YOLOv26{variant}")
    return max(float(row[METRIC_COLUMN]) for row in rows)

map50 = [best_map50(variant) for variant in VARIANTS]
print(dict(zip(MODEL_LABELS, map50)))

## 2. mAP@0.5 global por variante

In [ ]:
cores = ["#F4A261", "#2A9D8F", "#4C78A8", "#B565A7"]
hachuras = ["-", "//", "xx", "\\"]

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(MODEL_LABELS, map50, color=cores, edgecolor="black")
for barra, hachura, valor in zip(barras, hachuras, map50):
    barra.set_hatch(hachura)
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.01, f"{valor:.4f}", ha="center", va="bottom", fontsize=9)

ax.set_title("Comparação de mAP@0.5 - Modelos YOLOv26")
ax.set_ylabel("mAP@0.5")
ax.set_xlabel("Modelo YOLOv26")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()

## 3. mAP@0.5 por classe

Esta célula reavalia o melhor checkpoint armazenado de cada variante. A execução pode levar alguns minutos, mas mantém o gráfico por classe sincronizado com os pesos e o dataset reais.

In [ ]:
classes = ["cat", "civilian", "cow", "dog", "horse", "rescuer"]
per_class_map50 = {}

for variant in VARIANTS:
    checkpoint = artifact_path(variant, "weights/best.pt")
    metrics = YOLO(str(checkpoint)).val(data=str(DATA_YAML), imgsz=640, verbose=False)
    all_ap = np.asarray(metrics.box.all_ap)
    if all_ap.shape != (len(classes), 10):
        raise ValueError(f"Esperados {len(classes)} x 10 valores de AP; recebidos {all_ap.shape} para YOLOv26{variant}")
    per_class_map50[variant] = all_ap[:, 0]  # IoU = 0.5

x = np.arange(len(classes))
width = 0.2
fig, ax = plt.subplots(figsize=(11, 4.5))
for index, (variant, cor, hachura) in enumerate(zip(VARIANTS, cores, hachuras)):
    ax.bar(
        x + (index - 1.5) * width,
        per_class_map50[variant],
        width,
        label=f"YOLOv26 {variant.upper()}",
        color=cor,
        edgecolor="black",
        hatch=hachura,
    )

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("mAP@0.5")
ax.set_xlabel("Classe")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=4, frameon=False)
fig.tight_layout()
plt.show()